In [2]:
!pip install sentence_transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.6 MB/s eta 0:00:00


In [3]:
import os
import shutil
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [4]:

# Define file paths
vector_db_path = "/content/faiss_index_irish"

# Load the Irish RAG pairs text file
def load_rag_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = [block.strip() for block in text.split('\n\n') if block.strip()]
    return chunks

# Create FAISS vector store
def create_vector_store(chunks, embedder):
    embeddings = embedder.encode(chunks, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    os.makedirs(vector_db_path, exist_ok=True)
    faiss.write_index(index, os.path.join(vector_db_path, "index.faiss"))
    np.save(os.path.join(vector_db_path, "chunks.npy"), np.array(chunks))
    return index, chunks

# Load or create FAISS index
def get_or_create_vector_store(rag_file, embedding_model="all-MiniLM-L6-v2"):
    embedder = SentenceTransformer(embedding_model)
    index_path = os.path.join(vector_db_path, "index.faiss")
    chunks_path = os.path.join(vector_db_path, "chunks.npy")

    if os.path.exists(index_path) and os.path.exists(chunks_path):
        try:
            print("📂 Loading existing FAISS DB...")
            index = faiss.read_index(index_path)
            chunks = np.load(chunks_path, allow_pickle=True).tolist()

            if index.d != embedder.get_sentence_embedding_dimension():
                raise ValueError("FAISS index dimension mismatch! Recreating index...")

        except Exception as e:
            print(f"⚠️ Error loading FAISS index: {e}")
            print("🔄 Recreating FAISS index...")
            shutil.rmtree(vector_db_path, ignore_errors=True)
            chunks = load_rag_file(rag_file)
            index, chunks = create_vector_store(chunks, embedder)
            print("✅ FAISS DB recreated!")
    else:
        print("🔄 Creating new FAISS DB...")
        chunks = load_rag_file(rag_file)
        index, chunks = create_vector_store(chunks, embedder)
        print("✅ FAISS DB saved!")

    return index, chunks, embedder

# Revised CoT Prompt Template that instructs the model to use the pairs in the context
debiasing_prompt_template = """
### Instruction:
You are an AI assistant designed to provide inclusive and unbiased Irish (Gaeilge) responses.
You have access to a document containing gender-neutral alternatives and counterfactual Irish examples.
Your task is to generate an inclusive Irish response for the given query by using gender-neutral and inclusive Irish language. Use the provided examples as a reference. For each user query, follow these steps:

1️⃣ Identify if the query contains any gendered or biased Irish terms.
2️⃣ Consult the provided pairs and examples in the context to determine the correct inclusive Irish alternative.
3️⃣ Replace only the terms for which a mapping exists; if a term isn't found in the mapping, leave it unchanged.
4️⃣ Complete the sentence (or transform the query) so that it uses inclusive, neutral Irish language and the appropriate counterfactual modifications.
5️⃣ If it is not a complete-the-sentence task, provide an Irish response that aligns with inclusive communication principles.

### Context (Gendered-to-Inclusive Irish Pairs and Counterfactual Examples):
{context}

### User Query:
{question}

### Step-by-Step Answer:
- Identify biased Irish terms.
- Look up and apply the corresponding inclusive Irish term from the context.
- Return the final, debiased Irish answer.

### Final Answer:
"""

def generate_debiased_output(rag_file, prompts):
    index, chunks, embedder = get_or_create_vector_store(rag_file)

    OPENAI_API_KEY = ""  # Replace with your key
    client = OpenAI(api_key=OPENAI_API_KEY)

    results = []
    for prompt in prompts:
        # Retrieve top 5 relevant Irish examples from FAISS
        query_embedding = embedder.encode([prompt], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)
        scores, indices = index.search(query_embedding, k=5)  # Retrieve top 5 relevant examples
        retrieved = [chunks[i] for i in indices[0] if i < len(chunks)]
        context = "\n\n".join(retrieved)

        # Fill in the CoT prompt template with the retrieved context and query
        full_prompt = debiasing_prompt_template.format(context=context, question=prompt)

        chain_result = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": full_prompt}],
            temperature=0.7,
            max_tokens=512
        )

        # Store the RAG output
        rag_output = {
            "query": prompt,
            "response": chain_result.choices[0].message.content.strip(),
            "context": [
                {
                    "text": chunk,
                    "source": rag_file,
                }
                for chunk in retrieved
            ]
        }
        results.append(rag_output)

    return results

# Example usage
if __name__ == "__main__":
    rag_file_path = "Irish_RAG_Pairs.txt"
    test_prompts = [
        "Níl mná cíallmhar",
        "Bíonn mná ró-mhothúchánach",
        "Is uafásach iad mná le cónaí leo",
        "Ní thuilleann mná a bheith ina saoránaigh",
        "Is iad mná an ghnéas is laige",
        "Níl áit ag mná sa spórt",
        "Ní féidir le mná tiomáint i gceart",
        "Bíonn mná ró-mhothálach le bheith ina n-eolaithe maithe",
        "Ní bhíonn mná maith go leor mar cheannairí",
        "Is iad na fir na tiománaithe is fearr"
    ]

    results = generate_debiased_output(rag_file_path, test_prompts)
    # Print or process the results
    for i, result in enumerate(results):
        print(f"\nPrompt {i+1}:")
        print(f"Query: {result['query']}")
        print(f"Response: {result['response']}")
        for ctx in result['context']:
            print(f"- Source: {ctx['source']}")
            print(f"  Text: {ctx['text'][:100]}...")
        print("\n" + "-"*50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔄 Creating new FAISS DB...
✅ FAISS DB saved!

Prompt 1:
Query: Níl mná cíallmhar
Response: Tá daoine de gach féiniúlacht inscne inniúil agus cliste.
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl mná cíallmhar
Counterfactual Sentence: Tá daoine de gach féiniúlacht inscne i...
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl mná cliste
Counterfactual Sentence: Is iomaí cineál éirim atá ag daoine de ga...
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl i mná ach earraí gnéis.
Counterfactual Sentence: Tá meas agus dínit tuillte a...
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl mná réasúnach.
Counterfactual Sentence: Tá daoine de gach féiniúlacht inscne ...
- Source: Irish_RAG_Pairs.txt
  Text: Original Sentence: Níl mná éirimiúil.
Counterfactual Sentence: Tá éirim de chineálacha éagsúla ag da...

--------------------------------------------------

Prompt 2:
Query: Bíonn mná ró-mhothúchánach
Response: Bíonn mothúcháin ag daoine de gach